In [1]:
# Install core libs first
!pip install -q numpy>=2 pandas>=2.2.2

# Install ML + RAG stack
!pip install -q torch transformers faiss-cpu sentence-transformers

# Install LangChain (latest, no pinning)
!pip install -q langchain langchain-community

# Other utilities
!pip install -q streamlit pillow python-dotenv tavily-python neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 90.7 MB/s eta 0:00:00


In [2]:
import io
import numpy as np
import pandas as pd
from PIL import Image
from typing import TypedDict, Dict, List

In [3]:
docs = [
    "RAG stands for Retrieval Augmented Generation.",
    "LangGraph helps build agent-based workflows.",
    "Transformers are used in NLP tasks.",
    "Diabetes is treated using insulin therapy.",
    "Hypertension is controlled with diet and medication.",
    "AI systems can use retrieval to enhance generation."
]

In [4]:
class InputProcessor:
    def process(self, data):
        try:
            text = data.decode("utf-8")
            return {"type": "text", "text": text, "meta": {}}
        except:
            pass

        try:
            df = pd.read_csv(io.BytesIO(data))
            return {"type": "table", "text": str(df.head()), "meta": {"columns": list(df.columns)}}
        except:
            pass

        try:
            img = Image.open(io.BytesIO(data))
            return {"type": "image", "text": "Image uploaded", "meta": {"size": img.size}}
        except:
            pass

        return {"type": "unknown", "text": "", "meta": {}}

In [5]:
from sentence_transformers import SentenceTransformer
import faiss

class VectorStore:
    def __init__(self, docs):
        self.docs = docs
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.embeddings = self.model.encode(docs)

        dim = len(self.embeddings[0])
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(np.array(self.embeddings))

    def search(self, query, k=2):
        q_vec = self.model.encode([query])
        D, I = self.index.search(np.array(q_vec), k)
        return [self.docs[i] for i in I[0]]

vector_store = VectorStore(docs)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
class HybridRetriever:
    def retrieve(self, query):
        return vector_store.search(query)

In [9]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    max_length=256
)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

In [10]:
class PlannerAgent:
    def plan(self, query):
        prompt = f"Break this into steps: {query}"
        return llm(prompt)[0]["generated_text"]

class AnswerAgent:
    def generate(self, query, context):
        prompt = f"""
        Answer based on context:
        Context: {context}
        Question: {query}
        """
        return llm(prompt)[0]["generated_text"]

In [11]:
class Verifier:
    def check(self, answer):
        if len(answer.strip()) < 5:
            return 0.3
        return 0.9

In [12]:
class AgenticRAG:
    def __init__(self):
        self.retriever = HybridRetriever()
        self.planner = PlannerAgent()
        self.answerer = AnswerAgent()
        self.verifier = Verifier()

    def process(self, query):
        tasks = self.planner.plan(query)
        docs = self.retriever.retrieve(query)
        context = " ".join(docs)
        answer = self.answerer.generate(query, context)
        confidence = self.verifier.check(answer)

        return {
            "tasks": tasks,
            "retrieved": docs,
            "answer": answer,
            "confidence": confidence
        }

In [ ]:
import gradio as gr

rag = AgenticRAG()
processor = InputProcessor()

def rag_app(query, file):
    try:
        extra = ""

        if file is not None:
            data = file.read()
            processed = processor.process(data)
            extra = f"\n📂 File Meta: {processed['meta']}"
            query = processed["text"] or query

        result = rag.process(query)

        return f"""
Tasks:
{result['tasks']}

Retrieved:
{result['retrieved']}


Answer:
{result['answer']}

Confidence: {result['confidence']}
"""

    except Exception as e:
        return f"❌ Error: {str(e)}"

ui = gr.Interface(
    fn=rag_app,
    inputs=[
        gr.Textbox(label="Enter Query"),
        gr.File(label="Upload File (optional)")
    ],
     outputs=gr.Textbox(
        label="RAG Output",
        lines=20,        # 👈 visible height
        max_lines=40,    # 👈 scroll limit
        show_copy_button=True
    ),
    title="🤖 Agentic RAG (Multi Input)",
    description="Supports text + CSV + image input"
)

ui.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4f5797770c25c84538.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
